In [11]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

from utils.spark_session import get_spark
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType
from pyspark.sql.window import Window

spark = get_spark()

# Paths
SILVER_PATH     = '../data/silver'
GOLD_FACT       = '../data/gold/fact/hospital_financials'
GOLD_DIM_H      = '../data/gold/dims/dim_hospital'
GOLD_DIM_G      = '../data/gold/dims/dim_geography'
GOLD_DIM_T      = '../data/gold/dims/dim_time'
GOLD_DIM_P      = '../data/gold/dims/dim_payer'

df_silver = spark.read.format('delta').load(SILVER_PATH)

print(f'Silver rows loaded: {df_silver.count()}')
print(f'Silver schema:')
df_silver.printSchema()

Silver rows loaded: 25664
Silver schema:
root
 |-- ccn_key: string (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- fips: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- rucc_2013: double (nullable = true)
 |-- number_of_beds: double (nullable = true)
 |-- total_patient_revenue: double (nullable = true)
 |-- net_patient_revenue: double (nullable = true)
 |-- total_costs: double (nullable = true)
 |-- net_income: double (nullable = true)
 |-- less_total_operating_expense: double (nullable = true)
 |-- medicaid_charges: double (nullable = true)
 |-- inpatient_total_charges: double (nullable = true)
 |-- total_days_title_xviii: double (nullable = true)
 |-- total_days_title_xix: double (nullable = true)
 |-- cash_on_hand_and_in_banks: double (nullable = true)
 |-- total_current_assets: double (nullable = true)
 |-- total_current_liabilities: double (nullable = true)
 |-- depreciation_cost: double (nullable = true)
 |-- cost_of_charity_care: double

In [12]:
# ── Deduplicate Silver before building any Gold tables ────────────────────────
# Root cause: CMS source contains amended filings — some hospitals have two rows
# for the same fiscal year (original + amendment). Silver preserved all rows
# as received (correct Bronze/Silver behavior). Gold resolves to one row per
# (ccn_key, fiscal_year) by keeping the most complete record.

# Step 1: Score each row by counting non-null values across financial columns
FINANCIAL_COLS = [
    'total_patient_revenue', 'net_patient_revenue', 'total_costs',
    'net_income', 'less_total_operating_expense', 'medicaid_charges',
    'inpatient_total_charges', 'total_days_title_xviii', 'total_days_title_xix',
    'cash_on_hand_and_in_banks', 'total_current_assets', 'total_current_liabilities',
    'depreciation_cost', 'cost_of_charity_care', 'total_bad_debt_expense'
]

# Build a completeness score: count of non-null financial columns per row
# Each F.col(c).isNotNull().cast(IntegerType()) gives 1 if present, 0 if null
completeness_expr = sum(
    F.col(c).isNotNull().cast(IntegerType()) for c in FINANCIAL_COLS
)

df_silver_scored = df_silver.withColumn('_completeness', completeness_expr)

# Step 2: Rank rows within each (ccn_key, fiscal_year) group
# Row with highest completeness gets rank 1
# Ties broken by distress_label descending — if one row has the label and
# the other doesn't, we prefer to keep the labeled row
dedup_window = (
    Window
    .partitionBy('ccn_key', 'fiscal_year')
    .orderBy(
        F.col('_completeness').desc(),
        F.col('distress_label').desc()
    )
)

df_silver_deduped = (
    df_silver_scored
    .withColumn('_rank', F.row_number().over(dedup_window))
    .filter(F.col('_rank') == 1)
    .drop('_completeness', '_rank')
)

# Step 3: Verify
original_count  = df_silver.count()
deduped_count   = df_silver_deduped.count()
distinct_pairs  = df_silver_deduped.select('ccn_key', 'fiscal_year').distinct().count()

print(f'Silver original rows:   {original_count}')
print(f'Silver after dedup:     {deduped_count}')
print(f'Distinct (ccn, fy) pairs: {distinct_pairs}')
print(f'Rows removed:           {original_count - deduped_count}')
print(f'Any remaining dupes:    {deduped_count - distinct_pairs}')

# Step 4: Replace df_silver with the deduplicated version for all downstream cells
df_silver = df_silver_deduped

Silver original rows:   25664
Silver after dedup:     24634
Distinct (ccn, fy) pairs: 24634
Rows removed:           1030
Any remaining dupes:    0


In [13]:
# ── dim_time ─────────────────────────────────────────────────────────────────
# One row per distinct fiscal year.
# monotonically_increasing_id() generates a unique integer per row — not
# sequential, but guaranteed unique across partitions. Fine for a surrogate key.
dim_time = (
    df_silver
    .select('fiscal_year')
    .distinct()
    .withColumn('time_id', F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn(
        'fy_label',
        F.concat(F.lit('FY'), F.col('fiscal_year').cast(StringType()))
    )
    .orderBy('fiscal_year')
)

dim_time.write.format('delta').mode('overwrite').save(GOLD_DIM_T)
print(f'dim_time: {dim_time.count()} rows')
dim_time.show()


# ── dim_geography ─────────────────────────────────────────────────────────────
# One row per distinct (fips, state_code, rucc_2013) combination.
# Null-fips rows from unmatched counties are kept — they will produce a single
# null-fips dimension row; the fact table join will carry null geo_id for those
# hospitals rather than dropping them from the model.
dim_geography = (
    df_silver
    .select('fips', 'state_code', 'rucc_2013')
    .distinct()
    .withColumn('geo_id', F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn(
        'rural_level',
        F.when(F.col('rucc_2013') <= 3, 'Metro')
         .when(F.col('rucc_2013') <= 5, 'Micropolitan')
         .when(F.col('rucc_2013') <= 7, 'Small Town')
         .when(F.col('rucc_2013').isNotNull(), 'Remote Rural')
         .otherwise(None)   # null rucc stays null rather than mislabeled
    )
)

dim_geography.write.format('delta').mode('overwrite').save(GOLD_DIM_G)
print(f'dim_geography: {dim_geography.count()} rows')
dim_geography.show(5)


# ── dim_hospital ──────────────────────────────────────────────────────────────
# One row per distinct ccn_key.
# avg_beds averages across all fiscal years for that hospital — used purely
# for the static bed-size category label, not as a modeling feature.
dim_hospital = (
    df_silver
    .select('ccn_key', 'number_of_beds')
    .groupBy('ccn_key')
    .agg(F.avg('number_of_beds').alias('avg_beds'))
    .withColumn('hospital_id', F.monotonically_increasing_id().cast(IntegerType()))
    .withColumn(
        'bed_size_category',
        F.when(F.col('avg_beds') < 25,  'Critical Access')
         .when(F.col('avg_beds') < 100, 'Small Rural')
         .when(F.col('avg_beds') < 250, 'Medium Rural')
         .otherwise('Large Rural')
    )
)

dim_hospital.write.format('delta').mode('overwrite').save(GOLD_DIM_H)
print(f'dim_hospital: {dim_hospital.count()} unique hospitals')
dim_hospital.show(5)


# ── dim_payer ─────────────────────────────────────────────────────────────────
# One row per (ccn_key, fiscal_year) — payer mix changes year to year.
# Deviation from reference: no medicare_charges column, so medicare_day_pct
# is derived from inpatient day share rather than charge share.
# Deviation from reference: no commercial or uninsured charges column, so
# those fields are omitted.
dim_payer = (
    df_silver
    .select('ccn_key', 'fiscal_year',
            'medicaid_charges', 'total_patient_revenue',
            'total_days_title_xviii', 'total_days_title_xix',
            'cost_of_charity_care', 'total_bad_debt_expense')
    .withColumn(
        'medicaid_pct',
        F.when(F.col('total_patient_revenue') > 0,
               F.col('medicaid_charges') / F.col('total_patient_revenue')
        ).cast(DoubleType())
    )
    .withColumn(
        'medicare_day_pct',
        F.when(
            (F.col('total_days_title_xviii') + F.col('total_days_title_xix')) > 0,
            F.col('total_days_title_xviii') /
            (F.col('total_days_title_xviii') + F.col('total_days_title_xix'))
        ).cast(DoubleType())
    )
    .withColumn(
        'uncompensated_care_pct',
        F.when(F.col('total_patient_revenue') > 0,
               (F.col('cost_of_charity_care') + F.col('total_bad_debt_expense'))
               / F.col('total_patient_revenue')
        ).cast(DoubleType())
    )
    .withColumn('payer_id', F.monotonically_increasing_id().cast(IntegerType()))
    .select('payer_id', 'ccn_key', 'fiscal_year',
            'medicaid_pct', 'medicare_day_pct', 'uncompensated_care_pct')
)

dim_payer.write.format('delta').mode('overwrite').save(GOLD_DIM_P)
print(f'dim_payer: {dim_payer.count()} rows')
dim_payer.show(5)

dim_time: 14 rows
+-----------+-------+--------+
|fiscal_year|time_id|fy_label|
+-----------+-------+--------+
|       2011|     11|  FY2011|
|       2012|      8|  FY2012|
|       2013|      4|  FY2013|
|       2014|      5|  FY2014|
|       2015|      1|  FY2015|
|       2016|      9|  FY2016|
|       2017|     12|  FY2017|
|       2018|      0|  FY2018|
|       2019|      6|  FY2019|
|       2020|      7|  FY2020|
|       2021|     13|  FY2021|
|       2022|      3|  FY2022|
|       2023|      2|  FY2023|
|       2024|     10|  FY2024|
+-----------+-------+--------+

dim_geography: 1555 rows
+-----+----------+---------+------+------------+
| fips|state_code|rucc_2013|geo_id| rural_level|
+-----+----------+---------+------+------------+
|19071|        IA|      8.0|     0|Remote Rural|
|28011|        MS|      7.0|     1|  Small Town|
|30089|        MT|      8.0|     2|Remote Rural|
|39175|        OH|      7.0|     3|  Small Town|
|40101|        OK|      4.0|     4|Micropolitan|
+-----

In [14]:
# ── Window for year-over-year revenue change ──────────────────────────────────
# Partition by hospital so lag() only looks back within the same hospital's
# history. Order by fiscal_year so lag(1) is always the prior fiscal year.
hospital_window = Window.partitionBy('ccn_key').orderBy('fiscal_year')

df_features = (
    df_silver

    # FEATURE 1: Operating Margin
    # Core profitability from patient care operations.
    # Negative = hospital spends more on operations than it collects.
    # We use net_patient_revenue (actual collected) not total_patient_revenue
    # (gross charges), because gross charges include amounts never received.
    .withColumn(
        'operating_margin',
        F.when(
            F.col('net_patient_revenue') > 0,
            (F.col('net_patient_revenue') - F.col('less_total_operating_expense'))
            / F.col('net_patient_revenue')
        ).cast(DoubleType())
    )

    # FEATURE 2: Total Margin
    # Overall profitability including non-operating income (grants, investments).
    # A hospital can have a negative operating margin but survive on grants —
    # total margin captures whether that is happening.
    .withColumn(
        'total_margin',
        F.when(
            F.col('net_patient_revenue') > 0,
            F.col('net_income') / F.col('net_patient_revenue')
        ).cast(DoubleType())
    )

    # FEATURE 3: Cost-to-Revenue Ratio
    # Total costs as a fraction of net revenue collected.
    # Above 1.0 means the hospital spends more than it earns.
    .withColumn(
        'cost_to_revenue',
        F.when(
            F.col('net_patient_revenue') > 0,
            F.col('total_costs') / F.col('net_patient_revenue')
        ).cast(DoubleType())
    )

    # FEATURE 4: Medicaid Payer Mix %
    # Medicaid pays below cost for many services.
    # High medicaid_pct = structural margin pressure.
    .withColumn(
        'medicaid_pct',
        F.when(
            F.col('total_patient_revenue') > 0,
            F.col('medicaid_charges') / F.col('total_patient_revenue')
        ).cast(DoubleType())
    )

    # FEATURE 5: Medicare Day %
    # Share of inpatient days attributable to Medicare patients.
    # Denominator is Medicare + Medicaid days — our available proxy for
    # government payer volume. High % = concentrated government dependency.
    .withColumn(
        'medicare_day_pct',
        F.when(
            (F.col('total_days_title_xviii') + F.col('total_days_title_xix')) > 0,
            F.col('total_days_title_xviii')
            / (F.col('total_days_title_xviii') + F.col('total_days_title_xix'))
        ).cast(DoubleType())
    )

    # FEATURE 6: Occupancy Proxy
    # Government-payer inpatient days as a fraction of theoretical bed-capacity.
    # Not true occupancy (missing commercial/self-pay days), but a useful
    # utilization signal. Below 0.3 = very low volume relative to fixed costs.
    .withColumn(
        'occupancy_proxy',
        F.when(
            F.col('number_of_beds') > 0,
            (F.col('total_days_title_xviii') + F.col('total_days_title_xix'))
            / (F.col('number_of_beds') * 365.0)
        ).cast(DoubleType())
    )

    # FEATURE 7: Days Cash on Hand
    # How many days the hospital can pay its bills from available cash alone.
    # Below 30 days is the standard critical threshold used by rating agencies.
    .withColumn(
        'days_cash_on_hand',
        F.when(
            F.col('total_costs') > 0,
            F.col('cash_on_hand_and_in_banks') / (F.col('total_costs') / 365.0)
        ).cast(DoubleType())
    )

    # FEATURE 8: Current Ratio
    # Short-term liquidity: can the hospital cover obligations due within a year?
    # Below 1.0 = current liabilities exceed current assets = near-term distress.
    .withColumn(
        'current_ratio',
        F.when(
            F.col('total_current_liabilities') > 0,
            F.col('total_current_assets') / F.col('total_current_liabilities')
        ).cast(DoubleType())
    )

    # FEATURE 9: Uncompensated Care %
    # Charity care + bad debt as a share of total patient revenue.
    # High % = large volume of unpaid care draining cash reserves.
    .withColumn(
        'uncompensated_care_pct',
        F.when(
            F.col('total_patient_revenue') > 0,
            (F.col('cost_of_charity_care') + F.col('total_bad_debt_expense'))
            / F.col('total_patient_revenue')
        ).cast(DoubleType())
    )
)

# FEATURE 10: Year-over-Year Revenue Change
# Requires a lag — must be computed after the other withColumn() calls above
# because Window functions cannot be chained inside a single select/withColumn
# expression that also references newly created columns.
df_features = df_features.withColumn(
    'prior_year_revenue',
    F.lag('net_patient_revenue', 1).over(hospital_window)
)

df_features = df_features.withColumn(
    'revenue_yoy_change',
    F.when(
        F.col('prior_year_revenue') > 0,
        (F.col('net_patient_revenue') - F.col('prior_year_revenue'))
        / F.col('prior_year_revenue')
    ).cast(DoubleType())
)

# Verify: print summary stats for all 10 features
FEATURE_COLS = [
    'operating_margin', 'total_margin', 'cost_to_revenue',
    'medicaid_pct', 'medicare_day_pct', 'occupancy_proxy',
    'days_cash_on_hand', 'current_ratio',
    'uncompensated_care_pct', 'revenue_yoy_change'
]

print(f'Rows after feature engineering: {df_features.count()}')
df_features.select(FEATURE_COLS).describe().show()

Rows after feature engineering: 24634
+-------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+----------------------+-------------------+
|summary|    operating_margin|        total_margin|    cost_to_revenue|       medicaid_pct|    medicare_day_pct|     occupancy_proxy|  days_cash_on_hand|      current_ratio|uncompensated_care_pct| revenue_yoy_change|
+-------+--------------------+--------------------+-------------------+-------------------+--------------------+--------------------+-------------------+-------------------+----------------------+-------------------+
|  count|               24613|               24613|              24613|              24634|               24634|               24634|              24634|              23671|                 24634|              22424|
|   mean|-0.18909895825189033|-0.04953844730773289| 0.9987889205684584|0.18457497286105123|  0

In [15]:
# ── Join dimension surrogate keys onto the feature table ──────────────────────

# Join time_id
df_with_time = df_features.join(
    dim_time.select('fiscal_year', 'time_id'),
    on='fiscal_year',
    how='left'
)

# Join geo_id
# fips is nullable in Silver — left join means hospitals with null fips
# will receive null geo_id in the fact table. They are kept, not dropped.
df_with_geo = df_with_time.join(
    dim_geography.select('fips', 'state_code', 'rucc_2013', 'geo_id'),
    on=['fips', 'state_code', 'rucc_2013'],
    how='left'
)

# Join hospital_id
df_with_hospital = df_with_geo.join(
    dim_hospital.select('ccn_key', 'hospital_id'),
    on='ccn_key',
    how='left'
)

# Join payer_id
df_with_payer = df_with_hospital.join(
    dim_payer.select('ccn_key', 'fiscal_year', 'payer_id'),
    on=['ccn_key', 'fiscal_year'],
    how='left'
)

# ── Select final fact table columns ───────────────────────────────────────────
FEATURE_COLS = [
    'operating_margin', 'total_margin', 'cost_to_revenue',
    'medicaid_pct', 'medicare_day_pct', 'occupancy_proxy',
    'days_cash_on_hand', 'current_ratio',
    'uncompensated_care_pct', 'revenue_yoy_change'
]

fact_table = df_with_payer.select(
    # Surrogate keys (foreign keys to dimension tables)
    'hospital_id', 'time_id', 'geo_id', 'payer_id',
    # Natural keys kept for readability and debugging
    'ccn_key', 'fiscal_year',
    # Engineered feature columns
    *FEATURE_COLS,
    # Target variable
    'distress_label'
)

# ── Write Gold fact table ──────────────────────────────────────────────────────
(
    fact_table
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .save(GOLD_FACT)
)

print(f'Fact table written: {fact_table.count()} rows')
print(f'Distress rate in fact table: '
      f'{fact_table.filter("distress_label = 1").count() / fact_table.count():.4f}')

# Verify no rows were lost through the joins
print(f'\nNull surrogate key counts (all should be 0 or explained):')
fact_table.select(
    F.count(F.when(F.col('hospital_id').isNull(), 1)).alias('null_hospital_id'),
    F.count(F.when(F.col('time_id').isNull(),    1)).alias('null_time_id'),
    F.count(F.when(F.col('geo_id').isNull(),      1)).alias('null_geo_id'),
    F.count(F.when(F.col('payer_id').isNull(),    1)).alias('null_payer_id'),
).show()

Fact table written: 24634 rows
Distress rate in fact table: 0.0030

Null surrogate key counts (all should be 0 or explained):
+----------------+------------+-----------+-------------+
|null_hospital_id|null_time_id|null_geo_id|null_payer_id|
+----------------+------------+-----------+-------------+
|               0|           0|          0|            0|
+----------------+------------+-----------+-------------+



In [16]:
# ── Read back all five Gold tables from disk and verify ───────────────────────
# This confirms what was actually written, not just what is in memory.

fact   = spark.read.format('delta').load(GOLD_FACT)
d_time = spark.read.format('delta').load(GOLD_DIM_T)
d_geo  = spark.read.format('delta').load(GOLD_DIM_G)
d_hosp = spark.read.format('delta').load(GOLD_DIM_H)
d_pay  = spark.read.format('delta').load(GOLD_DIM_P)

print('=== GOLD LAYER SUMMARY ===')
print(f'fact_hospital_financials : {fact.count():>7} rows | '
      f'{len(fact.columns)} columns')
print(f'dim_time                 : {d_time.count():>7} rows')
print(f'dim_geography            : {d_geo.count():>7} rows')
print(f'dim_hospital             : {d_hosp.count():>7} rows')
print(f'dim_payer                : {d_pay.count():>7} rows')

print('\n=== FACT TABLE SCHEMA ===')
fact.printSchema()

print('\n=== LABEL DISTRIBUTION ===')
fact.groupBy('distress_label').count().orderBy('distress_label').show()

print('\n=== FISCAL YEAR COVERAGE ===')
fact.groupBy('fiscal_year').count().orderBy('fiscal_year').show()

print('\n=== SAMPLE FACT ROWS ===')
fact.select(
    'ccn_key', 'fiscal_year', 'operating_margin',
    'days_cash_on_hand', 'current_ratio', 'distress_label'
).show(5)

=== GOLD LAYER SUMMARY ===
fact_hospital_financials :   24634 rows | 17 columns
dim_time                 :      14 rows
dim_geography            :    1555 rows
dim_hospital             :    2197 rows
dim_payer                :   24634 rows

=== FACT TABLE SCHEMA ===
root
 |-- hospital_id: integer (nullable = true)
 |-- time_id: integer (nullable = true)
 |-- geo_id: integer (nullable = true)
 |-- payer_id: integer (nullable = true)
 |-- ccn_key: string (nullable = true)
 |-- fiscal_year: integer (nullable = true)
 |-- operating_margin: double (nullable = true)
 |-- total_margin: double (nullable = true)
 |-- cost_to_revenue: double (nullable = true)
 |-- medicaid_pct: double (nullable = true)
 |-- medicare_day_pct: double (nullable = true)
 |-- occupancy_proxy: double (nullable = true)
 |-- days_cash_on_hand: double (nullable = true)
 |-- current_ratio: double (nullable = true)
 |-- uncompensated_care_pct: double (nullable = true)
 |-- revenue_yoy_change: double (nullable = true)
 |-- 